# 03 — Modelo NCF Híbrido
Entrenamiento de Neural Collaborative Filtering con features de contenido (tags TF-IDF)

In [ ]:
# ── SETUP COLAB ──────────────────────────────────────────────
!git clone https://github.com/miguel-anay/music-recommender-ncf.git 2>/dev/null || echo "repo ya clonado"
!pip install -q "numpy<2" pandas torch scikit-learn matplotlib

!pip install -q gdown
!gdown --folder https://drive.google.com/drive/folders/1BfvqaOTlvb7EKtp8HuqgBReQ8PcUaUVe -O /content/lastfm --quiet

import sys, os
sys.path.append('/content/music-recommender-ncf/src')
os.makedirs('/content/music-recommender-ncf/results', exist_ok=True)
DATA_DIR_OVERRIDE = '/content/lastfm'
print("Setup completo")

In [ ]:
import sys
sys.path.append('../src')

import torch
import numpy as np
import matplotlib.pyplot as plt

from dataset import load_and_filter, binarize, build_tag_features, split_by_user, LastFMDataset
from model import NCFHybrid
from train import train
from config import EMBED_DIM, EPOCHS, BATCH_SIZE, LEARNING_RATE, NEG_RATIO

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {device}')

## Preparar datos (reproducir pipeline de notebook 02)

In [ ]:
df = binarize(load_and_filter())

user_ids      = sorted(df['userID'].unique())
artist_ids    = sorted(df['artistID'].unique())
user_to_idx   = {u: i for i, u in enumerate(user_ids)}
artist_to_idx = {a: i for i, a in enumerate(artist_ids)}

tag_matrix, _ = build_tag_features(artist_ids)
train_df, test_df = split_by_user(df)

train_dataset = LastFMDataset(train_df, tag_matrix, user_to_idx, artist_to_idx, neg_ratio=NEG_RATIO)

n_users   = len(user_ids)
n_artists = len(artist_ids)
print(f'Usuarios: {n_users} | Artistas: {n_artists} | Pares entrenamiento: {len(train_dataset):,}')

## Arquitectura del modelo

In [ ]:
model = NCFHybrid(n_users=n_users, n_artists=n_artists,
                  embed_dim=EMBED_DIM).to(device)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nParámetros entrenables: {total_params:,}')

## Entrenamiento

In [ ]:
history = train(model, train_dataset,
                epochs=EPOCHS, lr=LEARNING_RATE,
                batch_size=BATCH_SIZE, device=device)

## Curva de pérdida

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(range(1, len(history)+1), history, marker='o', linewidth=2)
plt.xlabel('Época')
plt.ylabel('BCELoss')
plt.title('Curva de pérdida — NCF Híbrido')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Loss inicial: {history[0]:.4f}')
print(f'Loss final:   {history[-1]:.4f}')
print(f'Reducción:    {(history[0]-history[-1])/history[0]*100:.1f}%')

## Predicciones de ejemplo

In [ ]:
model.eval()
sample_user = user_ids[0]
sample_u_idx = user_to_idx[sample_user]

# Predecir para los primeros 10 artistas
sample_artists = artist_ids[:10]
with torch.no_grad():
    u = torch.tensor([sample_u_idx] * 10, dtype=torch.long).to(device)
    a = torch.tensor([artist_to_idx[a] for a in sample_artists], dtype=torch.long).to(device)
    t = torch.tensor(tag_matrix[[artist_to_idx[a] for a in sample_artists]], dtype=torch.float32).to(device)
    scores = model(u, a, t).cpu().numpy()

print(f'Scores para usuario {sample_user} (primeros 10 artistas):')
for artist_id, score in sorted(zip(sample_artists, scores), key=lambda x: -x[1]):
    print(f'  Artista {artist_id}: {score:.4f}')

## Notas de arquitectura
- **Embeddings**: usuario (64d) + artista (64d) + proyección TF-IDF (50→64d)
- **MLP**: concat(192d) → 128 → 64 → 1 → Sigmoid
- **BCELoss** con 4 negativos aleatorios por positivo
- El mejor modelo se guarda en `results/best_model.pt`